# 3T quantitative figure

Combined subject-level comparison of simulation errors and LCModel fit rates. All markers show means and all error bars show sample SD across the same 12 validation subjects.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path('.').resolve()

SIMULATION_RESULTS = PROJECT_ROOT / 'results/HLSVD_L2_Comparison/3T_validation_clean_metabolite.h5'
CRLB_THRESHOLD = 20.0
SIMULATION_LOG_SCALE = True  # False for linear y-axes in panels a and b
FIGSIZE = (12.5, 4.0)
SAVE_PDF = "SavedGraphics/3T_quantitative.pdf"  # e.g. PROJECT_ROOT / 'results/PaperFigures/3T_quantitative.pdf'

METHODS = {
    'HLSVD + L2': 'LCModelHSVDL2',
    'WALRUS':     'LCModelAfterWalinet',
}
METABOLITES = {
    'tNAA': 'NAA_NAAG',
    'tCr':  'Cr_PCr',
    'tCho': 'GPC_PCh',
    'Glx':  'Glu_Gln',
}

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.lcmodel_quality import collect_fit_rates

subject_fit_rates = collect_fit_rates(
    project_root=PROJECT_ROOT,
    crlb_threshold=CRLB_THRESHOLD,
    methods=METHODS,
    metabolites=METABOLITES,
)

In [ ]:
from walinet.paper_figures.quantitative_3t import create_quantitative_figure

figure, simulation_subject_results = create_quantitative_figure(
    simulation_results=SIMULATION_RESULTS,
    subject_fit_rates=subject_fit_rates,
    simulation_log_scale=SIMULATION_LOG_SCALE,
    figsize=FIGSIZE,
)
if SAVE_PDF is not None:
    save_path = Path(SAVE_PDF).expanduser()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f'Saved: {save_path.resolve()}')

import matplotlib.pyplot as plt
plt.show()

In [ ]:
# Simulated input reference, independent of the evaluated method.
REFERENCE_METRIC = 'nuisance_l2_over_clean_metabolite_l2'
reference_by_subject = (
    simulation_subject_results[
        (simulation_subject_results['metric'] == REFERENCE_METRIC)
        & (simulation_subject_results['method'] == 'WALRUS')
    ]
    .set_index('subject')['mean']
    .sort_index()
)
reference_mean = reference_by_subject.mean()
reference_std = reference_by_subject.std(ddof=1)

print('Simulated input reference: ||n||_2 / ||m||_2')
print(f'Mean ± SD across {len(reference_by_subject)} subjects: '
      f'{reference_mean:.6g} ± {reference_std:.6g}')
print('\nSubject means:')
for subject, value in reference_by_subject.items():
    print(f'  {subject}: {value:.6g}')